In [2]:
import sqlite3
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load data
conn = sqlite3.connect("/Users/Maazuddin/Downloads/inventory.db")

df = pd.read_sql_query(
    """
    SELECT VendorNumber, Dollars, Quantity, Freight
    FROM vendor_invoice
    """,
    conn
)

conn.close()

df = df.dropna(
    subset=["VendorNumber", "Dollars", "Quantity", "Freight"]
)

X = df[["Dollars", "Quantity", "VendorNumber"]]
y = df["Freight"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# Model 1: Existing baseline
baseline = LinearRegression()
baseline.fit(
    X_train[["Dollars", "Quantity"]],
    y_train
)

baseline_predictions = baseline.predict(
    X_test[["Dollars", "Quantity"]]
)

# Model 2: Vendor-aware regression
preprocessor = ColumnTransformer(
    transformers=[
        (
            "vendor",
            OneHotEncoder(handle_unknown="ignore"),
            ["VendorNumber"]
        ),
        (
            "numeric",
            "passthrough",
            ["Dollars", "Quantity"]
        )
    ]
)

vendor_model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

vendor_model.fit(X_train, y_train)
vendor_predictions = vendor_model.predict(X_test)

# Evaluate both models
for name, predictions in [
    ("Baseline", baseline_predictions),
    ("Vendor-aware", vendor_predictions)
]:
    mae = mean_absolute_error(y_test, predictions)
    rmse = mean_squared_error(
        y_test, predictions
    ) ** 0.5
    r2 = r2_score(y_test, predictions)

    print(f"\n{name} Model")
    print(f"MAE:  {mae:.2f}")
    print(f"RMSE: {rmse:.2f}")
    print(f"R²:   {r2:.4f}")


Baseline Model
MAE:  24.46
RMSE: 124.43
R²:   0.9700

Vendor-aware Model
MAE:  24.46
RMSE: 124.43
R²:   0.9700


In [3]:
import numpy as np

print(
    "Maximum prediction difference:",
    np.max(
        np.abs(
            baseline_predictions - vendor_predictions
        )
    )
)

print(
    "Predictions identical:",
    np.allclose(
        baseline_predictions,
        vendor_predictions
    )
)

Maximum prediction difference: 5.3849362302571535e-08
Predictions identical: True


In [4]:
results = X_test[["VendorNumber", "Dollars", "Quantity"]].copy()

results["Actual_Freight"] = y_test
results["Baseline_Prediction"] = baseline_predictions
results["Vendor_Prediction"] = vendor_predictions

results["Baseline_Error"] = (
    results["Actual_Freight"]
    - results["Baseline_Prediction"]
).abs()

results["Vendor_Error"] = (
    results["Actual_Freight"]
    - results["Vendor_Prediction"]
).abs()

print(
    results.nlargest(10, "Baseline_Error")[
        [
            "VendorNumber",
            "Dollars",
            "Quantity",
            "Actual_Freight",
            "Baseline_Prediction",
            "Vendor_Prediction",
            "Baseline_Error",
            "Vendor_Error"
        ]
    ].round(2).to_string()
)

      VendorNumber     Dollars  Quantity  Actual_Freight  Baseline_Prediction  Vendor_Prediction  Baseline_Error  Vendor_Error
43            4425   185155.60     16075         4613.45               938.93             938.93         3674.52       3674.52
61            8004    57738.64      6283         1206.72               294.11             294.11          912.61        912.61
4313          3960  1018011.41    110018         4581.05              5093.92            5093.92          512.87        512.87
70            8673    21379.98      3008          571.07               111.07             111.07          460.00        460.00
1820          3960   900613.12     98001         4052.76              4505.79            4505.79          453.03        453.03
401           3960   823878.62     94829         4531.33              4112.02            4112.02          419.31        419.31
2018          3960   771864.94     86153         3473.39              3858.14            3858.14          384.7